In [13]:
import xarray as xr
import numpy as np
import pandas as pd
import os

In [14]:
folder = r"C:\Users\ajats\OneDrive\Desktop\north east\New folder"

tmean_file = os.path.join(
    folder,
    "2m_temperature_0_daily-mean.nc"
)

dew_file = os.path.join(
    folder,
    "2m_dewpoint_temperature_0_daily-mean.nc"
)

tmax_file = os.path.join(
    folder,
    "tmax.nc"
)

tmin_file = os.path.join(
    folder,
    "tmin.nc"
)

In [15]:
ds_tmean = xr.open_dataset(tmean_file)
ds_dew   = xr.open_dataset(dew_file)
ds_tmax  = xr.open_dataset(tmax_file)
ds_tmin  = xr.open_dataset(tmin_file)

print(ds_tmean)
print(ds_dew)
print(ds_tmax)
print(ds_tmin)

<xarray.Dataset> Size: 1MB
Dimensions:     (valid_time: 31, latitude: 84, longitude: 99)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 248B 2020-01-01 ... 2020-01-31
  * latitude    (latitude) float64 672B 29.8 29.7 29.6 29.5 ... 21.7 21.6 21.5
  * longitude   (longitude) float64 792B 88.0 88.1 88.2 88.3 ... 97.6 97.7 97.8
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 1MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-04T05:46 GRIB to CDM+CF via cfgrib-0.9.1...
<xarray.Dataset> Size: 1MB
Dimensions:     (valid_time: 31, latitude: 84, longitude: 99)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 248B 2020-01-01 ... 2020-01-31
  * latitude    (latit

In [16]:
Tmean = ds_tmean["t2m"]
Tdew  = ds_dew["d2m"]
Tmax  = ds_tmax["t2m"]
Tmin  = ds_tmin["t2m"]

In [17]:
def es_kelvin(Tk):

    return 6.108 * np.exp(
        (17.27 * (Tk - 273.15))
        /
        ((Tk - 273.15) + 237.3)
    )

In [18]:
es_mean = es_kelvin(Tmean)
ea_dew  = es_kelvin(Tdew)

RH = 100 * ea_dew / es_mean

RH = RH.clip(min=0, max=100)

RH.name = "RH_percent"

print(RH)

<xarray.DataArray 'RH_percent' (valid_time: 31, latitude: 84, longitude: 99)> Size: 1MB
array([[[21.873726, 20.751963, 19.602198, ..., 35.27949 , 34.991062,
         35.934326],
        [21.703835, 20.83899 , 20.354155, ..., 37.762356, 36.098392,
         36.411816],
        [20.153866, 19.591959, 19.53858 , ..., 41.780945, 39.496838,
         37.339462],
        ...,
        [      nan, 73.576004, 73.31645 , ..., 90.17373 , 90.87089 ,
         90.603645],
        [      nan,       nan, 75.37377 , ..., 89.13372 , 90.31371 ,
         90.13169 ],
        [      nan,       nan,       nan, ..., 88.13008 , 89.25802 ,
         89.71913 ]],

       [[35.83487 , 35.471203, 32.95256 , ..., 59.415363, 59.909397,
         61.40221 ],
        [35.79488 , 36.15958 , 34.82302 , ..., 61.244896, 60.47573 ,
         61.708187],
        [32.730167, 33.596592, 33.295475, ..., 64.8831  , 63.939354,
         63.105453],
...
        [      nan, 92.50599 , 92.94553 , ..., 56.481907, 59.604805,
         60.16

In [19]:
es_tmax = es_kelvin(Tmax)
es_tmin = es_kelvin(Tmin)

es_tmaxmin = (es_tmax + es_tmin) / 2

ea_tmaxmin_rh = es_tmaxmin * RH / 100

VPD3 = es_tmaxmin - ea_tmaxmin_rh

VPD3 = VPD3.clip(min=0)

VPD3.name = "VPD3_hPa"

print(VPD3)

<xarray.DataArray 'VPD3_hPa' (valid_time: 31, latitude: 84, longitude: 99)> Size: 1MB
array([[[ 2.2330499 ,  2.0198677 ,  1.7727137 , ...,  1.8590004 ,
          1.734866  ,  1.7943765 ],
        [ 1.7618263 ,  1.9189074 ,  1.9784284 , ...,  1.7394575 ,
          1.5704491 ,  1.6821818 ],
        [ 1.6761656 ,  2.1465795 ,  2.4838452 , ...,  1.6811126 ,
          1.5030296 ,  1.70658   ],
        ...,
        [        nan,  6.0992165 ,  6.121813  , ...,  2.1450748 ,
          1.9709148 ,  1.9201031 ],
        [        nan,         nan,  5.625496  , ...,  2.375595  ,
          2.1423492 ,  2.0482845 ],
        [        nan,         nan,         nan, ...,  2.4502869 ,
          2.3139477 ,  2.1372948 ]],

       [[ 1.7017697 ,  1.5252253 ,  1.36187   , ...,  1.3048127 ,
          1.2170782 ,  1.2416697 ],
        [ 1.3473563 ,  1.4423614 ,  1.4987795 , ...,  1.2317842 ,
          1.1176133 ,  1.1683918 ],
        [ 1.3253295 ,  1.6731267 ,  1.9437366 , ...,  1.1793346 ,
          1.04876

In [20]:
def clean_netcdf(ds):

    for var in ds.variables:

        ds[var].encoding.clear()

        ds[var].attrs.pop("_FillValue", None)
        ds[var].attrs.pop("missing_value", None)

    return ds

RH_ds = RH.to_dataset(name="RH_percent")
VPD3_ds = VPD3.to_dataset(name="VPD3_hPa")

RH_ds = clean_netcdf(RH_ds)
VPD3_ds = clean_netcdf(VPD3_ds)

RH_nc = os.path.join(
    folder,
    "RH_from_Tmean_Dewpoint.nc"
)

VPD_nc = os.path.join(
    folder,
    "VPD3_hPa.nc"
)

RH_ds.to_netcdf(RH_nc, engine="scipy")
VPD3_ds.to_netcdf(VPD_nc, engine="scipy")

print("NetCDF files saved")

NetCDF files saved


In [21]:
RH_excel = os.path.join(
    folder,
    "RH_from_Tmean_Dewpoint.xlsx"
)

VPD_excel = os.path.join(
    folder,
    "VPD3_hPa.xlsx"
)

RH.to_dataframe().reset_index().to_excel(
    RH_excel,
    index=False
)

VPD3.to_dataframe().reset_index().to_excel(
    VPD_excel,
    index=False
)

print("Excel files saved")

Excel files saved


In [22]:
RH_mean = RH.mean(
    dim=["latitude", "longitude"],
    skipna=True
)

VPD3_mean = VPD3.mean(
    dim=["latitude", "longitude"],
    skipna=True
)

RH_mean_file = os.path.join(
    folder,
    "RH_DailyMean.xlsx"
)

VPD_mean_file = os.path.join(
    folder,
    "VPD3_DailyMean.xlsx"
)

RH_mean.to_dataframe().reset_index().to_excel(
    RH_mean_file,
    index=False
)

VPD3_mean.to_dataframe().reset_index().to_excel(
    VPD_mean_file,
    index=False
)

print("Daily mean files saved")

Daily mean files saved


In [23]:
print("RH Mean:")
print(RH_mean)

print()

print("VPD3 Mean:")
print(VPD3_mean)

RH Mean:
<xarray.DataArray 'RH_percent' (valid_time: 31)> Size: 124B
array([68.776596, 76.680016, 83.419075, 84.3439  , 83.41412 , 71.57777 ,
       70.71114 , 76.16982 , 79.46784 , 76.17128 , 68.09909 , 65.588196,
       67.54873 , 72.26133 , 74.123   , 74.200134, 75.44982 , 73.78045 ,
       73.86071 , 72.85281 , 71.720566, 70.58846 , 71.19162 , 65.13395 ,
       59.185913, 61.635223, 63.5823  , 65.51882 , 68.808136, 75.20523 ,
       74.18127 ], dtype=float32)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 248B 2020-01-01 ... 2020-01-31
    number      int64 8B 0
Attributes: (12/27)
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      8316
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    GRIB_gridType:                            regular_ll
    ...                                       ...
    GRIB_missingValue:            